# E-Commerce Customer & Revenue Analysis

**Business question:** How can an online retailer grow revenue by improving repeat purchasing and focusing effort on the highest-value customers, markets, and products?

This notebook summarizes a reproducible SQL and Pandas analysis of 541,909 transaction lines from the UCI Online Retail dataset. The detailed SQL is stored in `sql/analysis.sql`.

In [ ]:
from pathlib import Path
import json
import sqlite3
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DB_PATH = ROOT / 'data' / 'processed' / 'ecommerce.db'
TABLE_DIR = ROOT / 'outputs' / 'tables'
FIGURE_DIR = ROOT / 'outputs' / 'figures'
summary = json.loads((ROOT / 'outputs' / 'summary_metrics.json').read_text())
pd.set_option('display.max_columns', 20)

## 1. Data quality and sales definition

A valid sale is a non-cancelled line with positive quantity and unit price, a customer ID, and a product description. The full source remains available for separate cancellation and quality analysis. The exclusion categories overlap and should not be added together.

In [ ]:
quality = pd.read_csv(TABLE_DIR / 'data_quality.csv').T
quality.columns = ['value']
quality.style.format('{:,.0f}')

## 2. Core performance

The customer-linked analysis contains **£8.91M in revenue, 18,562 orders, and 4,338 customers**. A 65.6% repeat-customer rate means repeat behavior is common, but 1,491 one-order customers still represent a clear conversion opportunity.

In [ ]:
kpis = pd.DataFrame({
    'Metric': ['Revenue', 'Orders', 'Customers', 'Average order value', 'Repeat customer rate'],
    'Value': [
        f"£{summary['revenue']:,.0f}",
        f"{summary['orders']:,}",
        f"{summary['customers']:,}",
        f"£{summary['average_order_value']:,.2f}",
        f"{summary['repeat_customer_rate_pct']:.1f}%",
    ]
})
kpis

## 3. SQL revenue trend

This query demonstrates a CTE, date transformation, aggregation, and `LAG` window function. The first and final months are partial, so complete-month comparisons focus on January through November 2011.

In [ ]:
monthly_sql = '''
WITH monthly AS (
    SELECT strftime('%Y-%m', invoice_date) AS order_month,
           SUM(order_revenue) AS revenue,
           COUNT(*) AS orders
    FROM orders
    GROUP BY strftime('%Y-%m', invoice_date)
), with_previous AS (
    SELECT *, LAG(revenue) OVER (ORDER BY order_month) AS prior_revenue
    FROM monthly
)
SELECT order_month, ROUND(revenue, 2) AS revenue, orders,
       ROUND(100.0 * (revenue - prior_revenue) / NULLIF(prior_revenue, 0), 2) AS mom_pct
FROM with_previous
ORDER BY order_month;
'''
with sqlite3.connect(DB_PATH) as conn:
    monthly = pd.read_sql_query(monthly_sql, conn)
monthly

In [ ]:
display(Image(filename=str(FIGURE_DIR / 'monthly_revenue.png'), width=900))

## 4. Customer behavior and concentration

Customers with 10 or more orders generated £4.57M. The highest customer revenue decile generated 61.4% of valid sales revenue, showing that retention of high-value and wholesale buyers is commercially important.

In [ ]:
frequency = pd.read_csv(TABLE_DIR / 'customer_purchase_frequency.csv')
frequency.assign(
    customer_share_pct=lambda x: 100 * x['customers'] / x['customers'].sum(),
    revenue_share_pct=lambda x: 100 * x['revenue'] / x['revenue'].sum(),
).round(2)

In [ ]:
display(Image(filename=str(FIGURE_DIR / 'revenue_concentration.png'), width=850))

## 5. RFM and cohort analysis

RFM segmentation identified 655 at-risk customers with £1.15M in historical revenue. Cohort retention drops after month zero, reinforcing the need for a structured first-to-second-order journey.

In [ ]:
segments = pd.read_csv(TABLE_DIR / 'rfm_segments.csv')
segments.style.format({
    'customers': '{:,.0f}', 'revenue': '£{:,.0f}',
    'revenue_per_customer': '£{:,.0f}', 'average_recency_days': '{:.1f}',
    'average_orders': '{:.1f}'
})

In [ ]:
display(Image(filename=str(FIGURE_DIR / 'cohort_retention.png'), width=950))

## 6. Market and product checks

Germany was the largest non-UK market and posted a 72.3% repeat-customer rate. Product analysis also surfaced a £168.5K item supported by only one order and one customer. I reported that record as an anomaly rather than allowing it to lead the stable product ranking.

In [ ]:
countries = pd.read_csv(TABLE_DIR / 'country_performance.csv')
products = pd.read_csv(TABLE_DIR / 'top_products.csv')
anomalies = pd.read_csv(TABLE_DIR / 'product_anomalies.csv')
display(countries.head(5), products.head(5), anomalies)

## Recommendations

1. **Test first-to-second-order messaging around days 30–45.** Measure incremental conversion and contribution margin against a holdout group.
2. **Prioritize the 655 at-risk customers by historical value.** Use differentiated offers instead of a blanket discount.
3. **Prepare inventory and campaigns before September.** Validate the apparent seasonal pattern with more years before using it as a forecast assumption.
4. **Protect top-decile relationships while reducing concentration risk.** Consider account support, early access, and replenishment reminders for high-value or wholesale buyers.
5. **Strengthen data collection and anomaly controls.** Capture return reasons, increase customer-ID coverage, and review high-value single-order product spikes.

## Limitations

The dataset contains no costs, marketing exposures, return reasons, or customer demographics. Customer analysis excludes rows without customer IDs. December 2010 and December 2011 are partial months. Results describe one retailer over roughly one year and do not establish causal effects.